# Fase 2 - Analise Exploratoria

Define alta gravidade e avalia contexto contra o alvo nos anos completos.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import (
    ANOS_COMPLETOS,
    COLUNAS_CONTEXTO,
    DADOS_DIR,
    FIGURAS_DIR,
    MIN_SUPPORT,
    PROCESSED_DIR,
    TABELAS_DIR,
)

plt.rcParams["figure.figsize"] = (11, 5)
sns.set_style("whitegrid")

In [ ]:
from scipy.stats import chi2_contingency
from src.data_loading import carregar_anos
from src.preparation import criar_desfecho_alta_gravidade, subset_com_vitimas

df, _ = carregar_anos(ignorar_ausentes=True)
df = df[df["ano"].isin(ANOS_COMPLETOS)].copy()
df_cv = subset_com_vitimas(df).copy()
df_cv["desfecho_alta_gravidade"] = criar_desfecho_alta_gravidade(df_cv)
print(df_cv["desfecho_alta_gravidade"].value_counts())
print((df_cv["desfecho_alta_gravidade"].value_counts(normalize=True) * 100).round(2))

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    return ((chi2 / n) / (min(r, k) - 1)) ** 0.5 if min(r, k) > 1 else 0

cv = pd.DataFrame([
    {"variavel": col, "cramers_v": cramers_v(df_cv[col], df_cv["desfecho_alta_gravidade"])}
    for col in COLUNAS_CONTEXTO
]).sort_values("cramers_v", ascending=False)
display(cv)

Hipoteses H1-H5: noite/rural, fim de semana, colisao frontal/atropelamento, clima e heterogeneidade urbano-rural.